# 🔍 Financial Crime Intelligence Copilot — Day 3
**Massive Scale Fraud Detection & AI Investigator Dashboard**

This notebook runs entirely in your Jupyter environment and features:
1. **Multi-Modal Ingestion:** Local files, Hugging Face Datasets (Streaming), and Synthetic data generation.
2. **Advanced Feature Engineering:** Time-window burst detection, rich schema parsing.
3. **Typology Classification & Alert Fatigue Reduction:** Maps raw alerts to typologies (e.g., Smurfing) and groups related events.
4. **AI Investigations & SAR:** Generates Suspicious Activity Reports using a local Hugging Face LLM (`Qwen/Qwen3-14B`).
5. **Interactive Copilot:** Ask questions about your flagged accounts.
6. **Relationship Graphs:** Visualizes transaction networks using `networkx`.

## 1. Setup & Imports
Install dependencies and import libraries.

In [ ]:
# 1. Setup & Imports
!pip install --quiet pandas numpy matplotlib seaborn gradio networkx pyvis scikit-learn transformers torch datasets accelerate

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
import networkx as nx
from pyvis.network import Network
import tempfile
import os
import time
from datetime import datetime, timedelta

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

print("✅ Setup complete. All libraries imported.")

## 2. Model Initialization
Load the local ROCm-optimized Hugging Face model. Includes fallback if the GPU is unavailable or OOM occurs.

In [ ]:
# 2. Model Initialization
MODEL_ID = "Qwen/Qwen2.5-72B-Instruct"
model_loaded = False
tokenizer = None
model = None

try:
    print(f"⏳ Attempting to load {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    model_loaded = True
    print("✅ Model loaded successfully on ROCm GPU!")
except Exception as e:
    print(f"⚠️ Could not load model {MODEL_ID}. Error: {e}")
    print("⚠️ Notebook will gracefully degrade to rule-based fallback mode for AI tasks.")


## 3. Data Loading & Streaming
Functions to generate synthetic data, load local files, or lazily stream massive datasets from Hugging Face.

In [ ]:
# 3. Data Loading & Synthetic Generation

def generate_synthetic_data(num_records=1000):
    """Fallback: Generates a rich synthetic dataset if no files are provided."""
    np.random.seed(42)
    now = datetime.now()
    
    accounts = [f"ACC_{i:04d}" for i in range(50)]
    recipients = [f"REC_{i:04d}" for i in range(30)]
    countries = ["USA", "UK", "Russia", "Nigeria", "North Korea", "Canada", "Japan"]
    
    data = []
    for i in range(num_records):
        acc = np.random.choice(accounts)
        # Introduce a burst anomaly
        if np.random.random() < 0.05:
            timestamp = now - timedelta(minutes=np.random.randint(1, 5))
            amount = np.random.randint(100_000, 2_000_000)
            country = np.random.choice(["Russia", "North Korea", "Nigeria"])
        else:
            timestamp = now - timedelta(days=np.random.randint(0, 30))
            amount = np.random.exponential(10000)
            country = np.random.choice(countries)
            
        data.append({
            "transaction_id": f"TXN_{i:06d}",
            "account_id": acc,
            "customer_name": f"Customer_{acc}",
            "amount": amount,
            "currency": "USD",
            "country": country,
            "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
            "transaction_type": np.random.choice(["WIRE", "ACH", "CARD"]),
            "merchant": f"Merchant_{np.random.randint(1, 100)}",
            "channel": np.random.choice(["MOBILE", "WEB", "BRANCH"]),
            "device_id": f"DEV_{np.random.randint(1000, 9999)}",
            "ip_address": f"192.168.1.{np.random.randint(1, 255)}",
            "recipient_account": np.random.choice(recipients)
        })
    
    df = pd.DataFrame(data)
    # Sort by time for realistic burst detection
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df

def stream_hf_dataset(dataset_id, max_records=5000):
    """Lazily streams records from a Hugging Face dataset."""
    try:
        print(f"Streaming dataset {dataset_id}...")
        ds = load_dataset(dataset_id, streaming=True, split="train")
        records = []
        for i, row in enumerate(ds):
            if i >= max_records: break
            records.append(row)
        return pd.DataFrame(records)
    except Exception as e:
        print(f"Failed to stream dataset: {e}")
        return pd.DataFrame()

def ingest_data(tx_path, cust_path, watch_path, hf_dataset_id):
    """Unified ingestion supporting local files, HF datasets, or synthetic generation."""
    df = pd.DataFrame()
    
    # 1. HF Dataset Priority
    if hf_dataset_id and hf_dataset_id.strip() != "":
        df = stream_hf_dataset(hf_dataset_id.strip())
        
    # 2. Local Files
    elif tx_path is not None:
        try:
            df = pd.read_csv(tx_path)
            if cust_path:
                df_cust = pd.read_csv(cust_path)
                if 'account_id' in df.columns and 'account_id' in df_cust.columns:
                    df = df.merge(df_cust, on="account_id", how="left")
            if watch_path:
                df_watch = pd.read_csv(watch_path)
                df_watch["watchlist_flag"] = True
                merge_col = "account_id" if "account_id" in df_watch.columns else "customer_name"
                if merge_col in df.columns:
                    df = df.merge(df_watch[[merge_col, "watchlist_flag"]], on=merge_col, how="left")
                    df["watchlist_flag"] = df["watchlist_flag"].fillna(False)
        except Exception as e:
            print(f"Error loading files: {e}")
            
    # 3. Fallback to Synthetic
    if df.empty:
        print("No valid input provided. Falling back to Synthetic Dataset generation.")
        df = generate_synthetic_data(2000)
        
    # Clean up schemas gracefully
    expected_cols = ["transaction_id", "account_id", "amount", "country", "timestamp", "recipient_account", "device_id"]
    for col in expected_cols:
        if col not in df.columns:
            df[col] = "UNKNOWN" if col != "amount" else 0
            
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce").fillna(pd.Timestamp("2000-01-01"))
    
    return df


## 4. Fraud Scoring & Typologies
Rule-based engine with burst detection, rich schema parsing, and alert fatigue reduction.

In [ ]:
# 4. Fraud Scoring, Burst Detection & Typologies

HIGH_RISK_COUNTRIES = ["Russia", "Nigeria", "North Korea"]

def detect_bursts(df):
    """Time-window burst detection: >3 transactions in 10 minutes."""
    df = df.sort_values(by=["account_id", "timestamp"])
    df["burst_flag"] = False
    
    # Calculate rolling count of transactions per account in 10min windows
    # This is a simplified vector approach: diff between current and 3rd previous
    df["time_diff"] = df.groupby("account_id")["timestamp"].diff(periods=3)
    df.loc[df["time_diff"] <= pd.Timedelta(minutes=10), "burst_flag"] = True
    return df

def score_transaction(row):
    score = 0
    typologies = []
    
    amt = row.get("amount", 0)
    if amt > 500_000: score += 50
    if amt > 1_000_000: score += 20
    
    country = str(row.get("country", ""))
    if country in HIGH_RISK_COUNTRIES: 
        score += 30
        typologies.append("Sanctions Risk")
        
    if row.get("burst_flag", False):
        score += 20
        typologies.append("Burst Activity / Smurfing")
        
    if row.get("watchlist_flag", False):
        score += 40
        typologies.append("Watchlist Match")
        
    if score >= 70:
        if "Smurfing" not in str(typologies): typologies.append("Money Laundering")
        
    if score == 0:
        typologies.append("Normal Activity")
        
    return min(score, 100), ", ".join(typologies)

def process_and_score(df):
    df = detect_bursts(df)
    results = df.apply(score_transaction, axis=1)
    df["risk_score"] = [r[0] for r in results]
    df["typology"] = [r[1] for r in results]
    
    df["risk_level"] = pd.cut(df["risk_score"], bins=[-1, 29, 69, 100], labels=["LOW", "MEDIUM", "HIGH"])
    return df

def group_alerts(df):
    """Alert fatigue reduction: groups raw transactions into unique investigation cases."""
    high_med = df[df["risk_level"].isin(["HIGH", "MEDIUM"])]
    if high_med.empty: return pd.DataFrame()
    
    # Group by account_id to create 'Cases'
    cases = high_med.groupby("account_id").agg(
        total_flagged_amount=("amount", "sum"),
        flagged_tx_count=("transaction_id", "count"),
        max_risk_score=("risk_score", "max"),
        primary_typology=("typology", lambda x: list(x)[0]),
        countries_involved=("country", lambda x: ", ".join(set(x)))
    ).reset_index()
    cases["case_id"] = ["CASE_" + str(i).zfill(4) for i in range(len(cases))]
    return cases.sort_values("max_risk_score", ascending=False)


## 5. Explainability & Counterfactuals
Generate rule-based explanations and "what-if" counterfactual analysis for flagged transactions.

In [ ]:
# 5. Explainability & Counterfactuals

def generate_rule_explanation(row):
    exp = ["**Risk Summary:** Transaction flagged based on rules."]
    reasons = []
    if row["amount"] > 500_000: reasons.append(f"High amount (${row['amount']})")
    if row["country"] in HIGH_RISK_COUNTRIES: reasons.append(f"High risk jurisdiction ({row['country']})")
    if row.get("burst_flag", False): reasons.append("Anomalous rapid burst of transactions detected.")
    
    exp.append("**Why it was flagged:** " + ", ".join(reasons) if reasons else "No specific red flags.")
    exp.append(f"**Typology Detected:** {row.get('typology', 'Unknown')}")
    exp.append("**Counterfactual Analysis:**")
    
    # Counterfactuals
    if row["amount"] > 500_000:
        exp.append(f"- If amount was < 500,000, score would drop by at least 50 points.")
    if row["country"] in HIGH_RISK_COUNTRIES:
        exp.append(f"- If country was not high-risk, score would drop by 30 points.")
        
    return "\n".join(exp)


## 6. Graph Visualization
Build relationship networks mapping accounts, recipients, and watchlists.

In [ ]:
# 6. Graph Visualization

def build_relationship_graph(df):
    """Builds a NetworkX graph and saves a PyVis interactive HTML file."""
    if df.empty: return None
    
    # Take top 100 high risk for visibility
    plot_df = df[df["risk_score"] > 30].head(100)
    if plot_df.empty: plot_df = df.head(100)
    
    G = nx.Graph()
    
    for _, row in plot_df.iterrows():
        acc = str(row["account_id"])
        rec = str(row.get("recipient_account", "UNKNOWN"))
        score = row["risk_score"]
        
        # Add nodes
        G.add_node(acc, group="Account", color="red" if score >= 70 else "orange")
        if rec != "UNKNOWN":
            G.add_node(rec, group="Recipient", color="blue")
            G.add_edge(acc, rec, weight=row["amount"], title=f"${row['amount']}")
            
    net = Network(height="500px", width="100%", bgcolor="#222222", font_color="white")
    net.from_nx(G)
    
    out_path = os.path.join(tempfile.gettempdir(), "fraud_network.html")
    net.save_graph(out_path)
    return out_path


## 7. LLM Reporting & SAR Generation
Generates complex SARs and handles the interactive Chatbot using the loaded model.

In [ ]:
# 7. LLM Reporting & SAR Generation

def query_llm(system_prompt, user_prompt, max_tokens=800):
    if not model_loaded:
        return "⚠️ LLM is offline (GPU/Memory limits). Operating in fallback rule-based mode."
        
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template is not None:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f"{system_prompt}\n\n{user_prompt}\n\nAnswer:\n"
        
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.3, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def generate_sar_report(case_data):
    if not model_loaded: return "⚠️ LLM Offline. SAR Generation requires the Hugging Face model."
    
    sys_prompt = """You are a Senior AML Investigator writing a Suspicious Activity Report (SAR).
Structure your response strictly as:
# Executive Summary
# Suspicious Activity Evidence
# Risk Assessment
# Recommended Action"""
    
    user_prompt = f"Please generate a SAR for the following aggregated case data:\n{case_data}"
    return query_llm(sys_prompt, user_prompt)

def investigator_chat(user_message, history, df):
    if df is None or df.empty: return "Please ingest and analyze data first."
    
    # Pass a summarized context to avoid OOM
    context = df.head(50).to_string()
    sys = f"You are an AI Financial Crime Copilot. Answer questions based ONLY on this dataset summary:\n{context}"
    
    return query_llm(sys, user_message, max_tokens=400)


## 8. Gradio UI Copilot Dashboard
Build the massive multi-tab dashboard interface.

In [ ]:
# 8. Gradio UI Copilot Dashboard

theme = gr.themes.Base(primary_hue="blue", neutral_hue="slate")
demo = gr.Blocks(theme=theme, title="FinCrime Copilot")

# States
state_df = gr.State()
state_cases = gr.State()

with demo:
    gr.Markdown("# 🛡️ Financial Crime Intelligence Copilot")
    
    with gr.Tabs():
        # --- TAB 1: Data Ingestion & Dashboard ---
        with gr.Tab("1. Ingestion & Executive Dashboard"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Data Sources")
                    tx_file = gr.File(label="Transactions CSV (Optional)")
                    cust_file = gr.File(label="Customers CSV (Optional)")
                    watch_file = gr.File(label="Watchlist CSV (Optional)")
                    hf_dataset = gr.Textbox(label="Hugging Face Dataset ID (e.g. 'synthesized-fraud-dataset', Optional)", placeholder="Leave blank for simulated data")
                    analyze_btn = gr.Button("🚀 Ingest & Analyze Data", variant="primary")
                    
                with gr.Column():
                    gr.Markdown("### Executive Summary")
                    dashboard_stats = gr.Markdown("No data analyzed yet.")
            
            results_table = gr.Dataframe(label="Raw Scored Transactions", interactive=False)
            
        # --- TAB 2: Alert Queue & Cases ---
        with gr.Tab("2. Alert Queue & Cases (Fatigue Reduction)"):
            gr.Markdown("### Grouped Investigation Cases")
            gr.Markdown("Raw alerts are grouped by Account ID and Time Windows to reduce investigator fatigue.")
            cases_table = gr.Dataframe(label="Unique Cases", interactive=False)
            
            with gr.Row():
                case_id_input = gr.Textbox(label="Enter Case ID to generate SAR")
                sar_btn = gr.Button("📄 Generate Official SAR (LLM)")
            sar_output = gr.Markdown(label="Suspicious Activity Report")
            
        # --- TAB 3: Counterfactuals & Explainability ---
        with gr.Tab("3. Explainability & Counterfactuals"):
            gr.Markdown("Select a row index from the raw transactions table to see why it was flagged and what would change the score.")
            with gr.Row():
                row_idx = gr.Number(label="Transaction Row Index", value=0, precision=0)
                explain_btn = gr.Button("🔍 Explain Risk Score")
            explain_output = gr.Markdown(label="Explainability Report")
            
        # --- TAB 4: Network Graph ---
        with gr.Tab("4. Relationship Graph"):
            gr.Markdown("Interactive visualization of high-risk transactions (Accounts -> Recipients).")
            graph_html = gr.HTML()
            
        # --- TAB 5: AI Copilot Chat ---
        with gr.Tab("5. AI Investigator Copilot"):
            gr.Markdown("Ask natural language questions about your analyzed dataset.")
            chatbot = gr.Chatbot(height=400)
            msg = gr.Textbox(label="Ask: e.g., 'Show transactions involving high-risk countries.'")
            clear = gr.ClearButton([msg, chatbot])
            
            def user_msg(user_message, history):
                return "", history + [[user_message, None]]
                
            def bot_msg(history, df):
                user_message = history[-1][0]
                bot_response = investigator_chat(user_message, history, df)
                history[-1][1] = bot_response
                return history
                
            msg.submit(user_msg, [msg, chatbot], [msg, chatbot], queue=False).then(
                bot_msg, [chatbot, state_df], chatbot
            )

    # Wiring Functions
    def run_pipeline(tx, cust, watch, hf):
        df = ingest_data(tx, cust, watch, hf)
        df = process_and_score(df)
        cases = group_alerts(df)
        
        # Stats
        total = len(df)
        high = (df["risk_level"] == "HIGH").sum()
        med = (df["risk_level"] == "MEDIUM").sum()
        stats = f"**Total Transactions:** {total} | **High Risk:** 🔴 {high} | **Medium Risk:** 🟡 {med}"
        
        graph_path = build_relationship_graph(df)
        graph_content = open(graph_path, "r").read() if graph_path else "Not generated"
        
        return df, cases, df.head(100), cases, stats, graph_content
        
    analyze_btn.click(
        fn=run_pipeline,
        inputs=[tx_file, cust_file, watch_file, hf_dataset],
        outputs=[state_df, state_cases, results_table, cases_table, dashboard_stats, graph_html]
    )
    
    def on_explain(idx, df):
        if df is None or df.empty: return "No data."
        try:
            row = df.iloc[int(idx)]
            return generate_rule_explanation(row)
        except: return "Invalid index."
        
    explain_btn.click(fn=on_explain, inputs=[row_idx, state_df], outputs=[explain_output])
    
    def on_sar(case_id, cases_df):
        if cases_df is None or cases_df.empty: return "No cases available."
        case_data = cases_df[cases_df["case_id"] == case_id]
        if case_data.empty: return "Case ID not found."
        return generate_sar_report(case_data.to_string())
        
    sar_btn.click(fn=on_sar, inputs=[case_id_input, state_cases], outputs=[sar_output])


## 9. Launch Application

In [ ]:
# 9. Launch Application
demo.launch(inline=True, share=False)